In [ ]:
import os
import ollama
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

In [ ]:
# Available ollama models locally


def _format_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


OllamaHost = os.getenv("OLLAMA_HOST")
OllamaBaseUrl = f"{OllamaHost}/v1"
OllamaClient = ollama.Client(host=OllamaHost)

available_models = [
    {
        "model": m.model,
        "size": _format_size(m.size or 0),
        "params": m.details.parameter_size if m.details else "—",
        "quant": m.details.quantization_level if m.details else "—",
        "modified": m.modified_at.strftime("%Y-%m-%d %H:%M") if m.modified_at else "—",
    }
    for m in OllamaClient.list().models
]

rows = "\n".join(f"{m['model']}" for m in available_models)
print(rows)

In [ ]:
model_names = [m["model"] for m in available_models]
DEFAULT_MODEL = (
    "gpt-oss:20b"
    if "gpt-oss:20b" in model_names
    else (model_names[0] if model_names else "gpt-oss:20b")
)

In [ ]:
system_message = """
You are DataAlchemy, a synthetic data designer.

You do NOT write data rows yourself. Instead you design a precise generation SPECIFICATION that a downstream program uses to synthesize realistic rows programmatically. Because the program generates the actual values, you can describe datasets of any size — never refuse or apologize based on the number of rows.

Given requirements (topic, complexity, format, row_count), return a JSON spec describing the tables, their columns, and how many rows each table should have.

Rules:
- Design table-shaped, relational data. Simple: 1 table. Medium: at least 2 related tables. Hard: at least 3 related tables with realistic primary-key and foreign-key relationships.
- Distribute row_count across tables so the per-table "rows" values sum to approximately row_count. Lookup/parent tables get fewer rows; child/transaction tables get more.
- Every table needs exactly one "id" column as its primary key. Child tables reference parents with "foreign_key" columns.
- Choose realistic column names and types for the topic. Provide value pools for categorical fields and sensible min/max and date ranges.

Each column has a "name" and a "type". "type" must be one of:
  id            -> unique sequential primary key. optional "prefix" (e.g. "CUST").
  foreign_key   -> references a parent table's id. required "references": "table.id_column".
  name | first_name | last_name | email | username | phone | company | job
  address | street_address | city | state | postcode | country | country_code
  int           -> requires "min" and "max".
  float         -> requires "min" and "max"; optional "decimals" (default 2).
  bool
  date          -> requires "start" and "end" as "YYYY-MM-DD". optional "after"/"min_days"/"max_days" (see below).
  datetime      -> like date, with a time component.
  category      -> requires "values": [...]; optional "weights": [...] (same length).
  formula       -> a value computed from other columns IN THE SAME ROW. requires "expr".
                   "expr" may use other column names and + - * / // % ** and round(), min(), max(), abs().
                   optional "decimals". Example: {"name": "line_total", "type": "formula", "expr": "quantity * unit_price", "decimals": 2}
  sentence | text
  uuid

Cross-field realism (IMPORTANT — prefer these over loosely-related independent columns):
- Person fields in a row are automatically coherent: "email" and "username" are derived from "name". Just include the columns; do not try to match them yourself.
- Geographic fields in a row (name, phone, street_address, city, state, postcode, country, country_code) are all drawn from ONE locale, so they are mutually consistent. To control which countries appear, set a table-level "locales" array of Faker locale codes, e.g. "locales": ["en_US", "en_GB", "de_DE", "fr_FR"]. Each row randomly uses one locale. Do NOT model country as a free "category" when you want the address to match — use "locales" instead.
- For totals and other computed numbers, use a "formula" column (e.g. line_total = quantity * unit_price) instead of an independent random number.
- For sequential dates (e.g. a ship_date that must follow an order_date), give the later column "after": "<earlier_date_column>" with optional "min_days"/"max_days".

Always return exactly one valid JSON object. Do not include Markdown fences, comments, or any text outside the JSON.
The status value must be exactly one of: "success" or "error".

Response schema:
{
    "status": "success",
    "reply": "Designed the dataset.",
    "data": {
        "description": "What the dataset represents and how the tables relate.",
        "tables": [
            {
                "name": "customers",
                "rows": 100,
                "locales": ["en_US", "en_GB", "de_DE"],
                "columns": [
                    {"name": "customer_id", "type": "id", "prefix": "CUST"},
                    {"name": "full_name", "type": "name"},
                    {"name": "email", "type": "email"},
                    {"name": "city", "type": "city"},
                    {"name": "country", "type": "country"},
                    {"name": "signup_date", "type": "date", "start": "2021-01-01", "end": "2024-12-31"},
                    {"name": "lifetime_value", "type": "float", "min": 0, "max": 50000, "decimals": 2}
                ]
            },
            {
                "name": "orders",
                "rows": 400,
                "columns": [
                    {"name": "order_id", "type": "id", "prefix": "ORD"},
                    {"name": "customer_id", "type": "foreign_key", "references": "customers.customer_id"},
                    {"name": "order_date", "type": "date", "start": "2021-01-01", "end": "2024-12-31"},
                    {"name": "ship_date", "type": "date", "after": "order_date", "min_days": 1, "max_days": 21},
                    {"name": "quantity", "type": "int", "min": 1, "max": 10},
                    {"name": "unit_price", "type": "float", "min": 5, "max": 500, "decimals": 2},
                    {"name": "order_total", "type": "formula", "expr": "quantity * unit_price", "decimals": 2},
                    {"name": "status", "type": "category", "values": ["pending", "shipped", "delivered", "cancelled"], "weights": [1, 2, 5, 1]}
                ]
            }
        ]
    },
    "error": {
        "message": "A clear explanation of what went wrong and how the user can retry."
    }
}

Include only the fields that apply to the current status: data for "success" and error for "error".
"""

checker_message = """
You extract requirements for a synthetic dataset request.

Return exactly one valid JSON object with this schema:
{
    "topic": "dataset subject or null",
    "complexity": "Simple, Medium, Hard, or null",
    "format": "CSV, JSON, Markdown, or null",
    "rows": "total number of rows as an integer, or null",
    "missing": ["topic", "complexity", "format", "rows"],
    "reply": "short user-facing reply"
}

Rules:
- Use the provided current state unless the latest user message overrides it.
- Extract complexity only if the user explicitly wrote Simple, Medium, or Hard. Do not infer it from the topic.
- Extract format only if the user explicitly wrote CSV, JSON, or Markdown. Do not infer it from the topic.
- Extract rows only if the user explicitly stated a total number of rows, items, records, or entries. Set rows to a plain integer with no commas or words. Do not infer a count from the topic.
- If there is no clear dataset subject, set topic to null.
- Include every still-missing field in missing.
- If topic is missing, ask the user to type the dataset topic.
- If only complexity, format, or rows is missing, keep reply short because the app will show option cards.
- Do not generate data.
"""

In [ ]:
import ast
import csv as csvmod
import datetime as dt
import io
import json
import operator
import random
import re
import tempfile
import threading
import unicodedata
import zipfile
from pathlib import Path

from faker import Faker

openai = OpenAI(base_url=OllamaBaseUrl, api_key="ollama")

REQUIRED_FIELDS = ("topic", "complexity", "format", "rows")
COMPLEXITY_OPTIONS = ("Simple", "Medium", "Hard")
FORMAT_OPTIONS = ("CSV", "JSON", "Markdown")
ROW_OPTIONS = ("10", "50", "100", "500")
MAX_ROWS_PER_TABLE = 100_000
DEFAULT_LOCALES = (
    "en_US",
    "en_GB",
    "en_CA",
    "en_AU",
    "de_DE",
    "fr_FR",
    "es_ES",
    "it_IT",
)


def explicit_complexity(message):
    match = re.search(r"\b(simple|medium|hard)\b", message, re.IGNORECASE)
    return match.group(1).title() if match else None


def explicit_format(message):
    match = re.search(r"\b(csv|json|markdown|md)\b", message, re.IGNORECASE)
    if not match:
        return None
    value = match.group(1).lower()
    return "Markdown" if value in {"markdown", "md"} else value.upper()


def explicit_rows(message):
    match = re.search(
        r"\b(\d+)\s*(rows?|items?|records?|entries|entry)\b", message, re.IGNORECASE
    )
    return int(match.group(1)) if match else None


def required_file_count(complexity):
    return {"Simple": 1, "Medium": 2, "Hard": 3}.get(complexity, 1)


# --- Programmatic dataset synthesis -------------------------------------------------
# The model returns a spec (tables -> columns + per-table row counts); we generate the
# actual rows here so any volume is produced reliably, with referential integrity AND
# cross-field coherence:
#   * each row picks one locale, so name/address/city/country/phone agree;
#   * email/username are derived from the row's generated name;
#   * "formula" columns are computed from sibling columns (e.g. quantity * unit_price);
#   * "after" date columns are generated relative to an earlier date in the same row.

_faker_cache = {}
ENTITY_TYPES = {
    "name",
    "first_name",
    "last_name",
    "email",
    "username",
    "phone",
    "company",
    "job",
    "address",
    "street_address",
    "city",
    "state",
    "postcode",
    "country",
    "country_code",
    "sentence",
    "text",
    "uuid",
}
_SAFE_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
}
_SAFE_FUNCS = {
    "round": round,
    "min": min,
    "max": max,
    "abs": abs,
    "int": int,
    "float": float,
}


def _safe_name(name, fallback="table"):
    return re.sub(r"[^a-zA-Z0-9_-]+", "_", str(name)).strip("_") or fallback


def _ascii(text):
    return "".join(
        c
        for c in unicodedata.normalize("NFKD", str(text))
        if not unicodedata.combining(c)
    )


def _localized_faker(locale):
    instance = _faker_cache.get(locale)
    if instance is None:
        try:
            instance = Faker(locale)
        except Exception:
            instance = Faker()
        _faker_cache[locale] = instance
    return instance


def _row_context(table):
    """Per-row context: one locale (so geography agrees) and a lazy person profile."""
    locales = table.get("locales") or list(DEFAULT_LOCALES)
    if isinstance(locales, str):
        locales = [locales]
    locale = random.choice(locales) if locales else random.choice(DEFAULT_LOCALES)
    return {"locale": locale, "faker": _localized_faker(locale)}


def _person(ctx):
    """Generate (once) the row's person so name/email/username stay consistent."""
    if "name" not in ctx:
        faker = ctx["faker"]
        first, last = faker.first_name(), faker.last_name()
        ctx["first_name"] = first
        ctx["last_name"] = last
        ctx["name"] = f"{first} {last}"
    return ctx


def _random_date(col):
    try:
        start = dt.date.fromisoformat(str(col.get("start") or "2020-01-01"))
        end = dt.date.fromisoformat(str(col.get("end") or "2024-12-31"))
    except ValueError:
        start, end = dt.date(2020, 1, 1), dt.date(2024, 12, 31)
    if end < start:
        start, end = end, start
    return start + dt.timedelta(days=random.randint(0, max((end - start).days, 0)))


def _entity_value(type_name, ctx):
    faker = ctx["faker"]
    if type_name in {"name", "first_name", "last_name"}:
        _person(ctx)
        return ctx["name"] if type_name == "name" else ctx[type_name]
    if type_name == "username":
        _person(ctx)
        return (
            _ascii(f"{ctx['first_name']}.{ctx['last_name']}").lower().replace(" ", "")
        )
    if type_name == "email":
        _person(ctx)
        try:
            domain = faker.free_email_domain()
        except Exception:
            domain = "example.com"
        local = (
            _ascii(f"{ctx['first_name']}.{ctx['last_name']}").lower().replace(" ", "")
        )
        return f"{local}@{domain}"

    simple = {
        "phone": faker.phone_number,
        "company": faker.company,
        "job": getattr(faker, "job", faker.word),
        "address": lambda: faker.address().replace("\n", ", "),
        "street_address": getattr(faker, "street_address", faker.address),
        "city": faker.city,
        "state": getattr(faker, "state", lambda: ""),
        "postcode": getattr(faker, "postcode", lambda: ""),
        "country": getattr(faker, "current_country", faker.country),
        "country_code": getattr(faker, "current_country_code", faker.country_code),
        "sentence": lambda: faker.sentence(nb_words=8),
        "text": lambda: faker.text(max_nb_chars=120),
        "uuid": faker.uuid4,
    }
    provider = simple.get(type_name)
    try:
        return provider() if provider else faker.word()
    except Exception:
        return faker.word()


def _make_value(col, fk_pools, ctx):
    type_name = str(col.get("type") or "text").lower()

    if type_name == "int":
        lo, hi = int(col.get("min", 0)), int(col.get("max", 100))
        lo, hi = min(lo, hi), max(lo, hi)
        return random.randint(lo, hi)
    if type_name == "float":
        lo, hi = float(col.get("min", 0)), float(col.get("max", 1))
        lo, hi = min(lo, hi), max(lo, hi)
        return round(random.uniform(lo, hi), int(col.get("decimals", 2)))
    if type_name == "bool":
        return random.choice([True, False])
    if type_name in {"date", "datetime"}:
        day = _random_date(col)
        if type_name == "date":
            return day.isoformat()
        clock = dt.time(
            random.randint(0, 23), random.randint(0, 59), random.randint(0, 59)
        )
        return dt.datetime.combine(day, clock).isoformat(sep=" ")
    if type_name == "category":
        values = col.get("values") or ["A", "B", "C"]
        weights = col.get("weights")
        if weights and len(weights) == len(values):
            return random.choices(values, weights=weights, k=1)[0]
        return random.choice(values)
    if type_name == "foreign_key":
        ref = str(col.get("references") or "")
        pool = fk_pools.get(ref) or fk_pools.get(ref.split(".")[0])
        return random.choice(pool) if pool else None
    if type_name in ENTITY_TYPES:
        return _entity_value(type_name, ctx)
    return _entity_value("text", ctx)


def _date_after(base_value, col, type_name):
    try:
        base_date = dt.date.fromisoformat(str(base_value)[:10])
    except (ValueError, TypeError):
        base_date = dt.date.today()
    lo, hi = int(col.get("min_days", 1)), int(col.get("max_days", 30))
    lo, hi = min(lo, hi), max(lo, hi)
    new_date = base_date + dt.timedelta(days=random.randint(lo, hi))
    if type_name == "datetime":
        clock = dt.time(
            random.randint(0, 23), random.randint(0, 59), random.randint(0, 59)
        )
        return dt.datetime.combine(new_date, clock).isoformat(sep=" ")
    return new_date.isoformat()


def _safe_eval(expr, variables):
    """Evaluate an arithmetic formula over row values; unknown names resolve to 0."""

    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.Name):
            value = variables.get(node.id, 0)
            return value if isinstance(value, (int, float)) else 0
        if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_BINOPS:
            return _SAFE_BINOPS[type(node.op)](
                evaluate(node.left), evaluate(node.right)
            )
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = evaluate(node.operand)
            return value if isinstance(node.op, ast.UAdd) else -value
        if (
            isinstance(node, ast.Call)
            and isinstance(node.func, ast.Name)
            and node.func.id in _SAFE_FUNCS
            and not node.keywords
        ):
            return _SAFE_FUNCS[node.func.id](*[evaluate(arg) for arg in node.args])
        raise ValueError("unsupported expression")

    return evaluate(ast.parse(expr, mode="eval").body)


def _resolve_deferred(row, deferred):
    """Resolve formula/`after` columns once their dependencies exist (multi-pass)."""
    pending = list(deferred)
    for _ in range(len(pending) + 1):
        if not pending:
            break
        still, progressed = [], False
        for col in pending:
            name = col.get("name")
            type_name = str(col.get("type")).lower()
            if type_name in {"date", "datetime"}:
                base = row.get(col.get("after"))
                if base is None:
                    still.append(col)
                    continue
                row[name] = _date_after(base, col, type_name)
                progressed = True
            else:  # formula
                expr = str(col.get("expr") or "0")
                refs = [
                    n for n in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expr) if n in row
                ]
                if any(row.get(n) is None for n in refs):
                    still.append(col)
                    continue
                try:
                    value = _safe_eval(expr, row)
                    if "decimals" in col and isinstance(value, (int, float)):
                        value = round(value, int(col["decimals"]))
                    row[name] = value
                except Exception:
                    row[name] = None
                progressed = True
        pending = still
        if not progressed:
            break


def _table_order(tables):
    """Order tables so a table is generated after any parent it references."""
    by_name = {t.get("name"): t for t in tables}
    ordered, visited = [], set()

    def visit(table, stack):
        name = table.get("name")
        if name in visited or name in stack:
            return
        stack.add(name)
        for col in table.get("columns", []):
            if str(col.get("type")).lower() == "foreign_key":
                parent = str(col.get("references") or "").split(".")[0]
                if parent in by_name:
                    visit(by_name[parent], stack)
        stack.discard(name)
        visited.add(name)
        ordered.append(table)

    for table in tables:
        visit(table, set())
    return ordered


def synthesize_tables(tables):
    """Turn a spec into {table_name: {"columns": [...], "rows": [ {..}, ... ]}}."""
    fk_pools = {}
    rendered = {}
    for table in _table_order(tables):
        name = _safe_name(table.get("name"))
        columns = table.get("columns") or []
        col_names = [c.get("name") or f"col_{i}" for i, c in enumerate(columns)]
        count = max(1, min(int(table.get("rows", 10) or 10), MAX_ROWS_PER_TABLE))

        rows = []
        for index in range(1, count + 1):
            ctx = _row_context(table)
            row = {}
            deferred = []
            for col in columns:
                col_name = col.get("name")
                type_name = str(col.get("type") or "text").lower()
                if type_name == "id":
                    prefix = col.get("prefix")
                    row[col_name] = f"{prefix}-{index:06d}" if prefix else index
                elif type_name == "formula" or (
                    type_name in {"date", "datetime"} and col.get("after")
                ):
                    row[col_name] = None
                    deferred.append(col)
                else:
                    row[col_name] = _make_value(col, fk_pools, ctx)
            _resolve_deferred(row, deferred)
            rows.append(row)

        for col in columns:
            if str(col.get("type")).lower() == "id":
                key = f"{table.get('name')}.{col.get('name')}"
                fk_pools[key] = [row[col.get("name")] for row in rows]

        rendered[name] = {"columns": col_names, "rows": rows}
    return rendered


def _format_cell(value):
    if value is None:
        return ""
    if isinstance(value, bool):
        return "true" if value else "false"
    return str(value)


def _to_csv(columns, rows):
    buffer = io.StringIO()
    writer = csvmod.writer(buffer)
    writer.writerow(columns)
    for row in rows:
        writer.writerow([_format_cell(row.get(c)) for c in columns])
    return buffer.getvalue()


def _to_markdown(columns, rows):
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for row in rows:
        cells = [_format_cell(row.get(c)).replace("|", "\\|") for c in columns]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines) + "\n"


def render_files(rendered, dataset_format):
    extension = {"CSV": "csv", "JSON": "json", "Markdown": "md"}.get(
        dataset_format, "csv"
    )
    files = []
    for name, table in rendered.items():
        columns, rows = table["columns"], table["rows"]
        if dataset_format == "JSON":
            content = json.dumps(rows, indent=2, default=str)
        elif dataset_format == "Markdown":
            content = _to_markdown(columns, rows)
        else:
            content = _to_csv(columns, rows)
        files.append({"filename": f"{name}.{extension}", "content": content})
    return files


def call_llm_json(messages, model):
    result = (
        openai.chat.completions.create(
            model=model,
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=8192,
        )
        .choices[0]
        .message.content
    )
    text = result.strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        return json.loads(text[start : end + 1])


def make_zip(files):
    output_dir = Path(tempfile.mkdtemp(prefix="dataalchemy_"))
    zip_path = output_dir / "dataalchemy_dataset.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for index, file_info in enumerate(files or [], start=1):
            filename = Path(
                str(file_info.get("filename") or f"dataset_{index}.txt")
            ).name
            content = file_info.get("content", "")
            archive.writestr(filename, str(content))
    return str(zip_path)


def next_question_card(state):
    """Return the option card for the next missing field, or None if complete."""
    if not state.get("complexity"):
        return gr.ChatMessage(
            role="assistant",
            content="How complex should the dataset be?",
            options=[
                {"label": option, "value": f"complexity:{option}"}
                for option in COMPLEXITY_OPTIONS
            ],
        )
    if not state.get("format"):
        return gr.ChatMessage(
            role="assistant",
            content="Which format would you like?",
            options=[
                {"label": option, "value": f"format:{option}"}
                for option in FORMAT_OPTIONS
            ],
        )
    if not state.get("rows"):
        return gr.ChatMessage(
            role="assistant",
            content="How many rows (or items) in total? Pick one or type a number.",
            options=[
                {"label": option, "value": f"rows:{option}"} for option in ROW_OPTIONS
            ],
        )
    return None


def _spinner_message(caption):
    """A 'thinking' bubble with a live spinner and a single static caption."""
    return gr.ChatMessage(
        role="assistant",
        content="",
        metadata={"title": caption, "status": "pending"},
    )


def stream_thinking(chat, holder, work_fn, caption):
    """Run work_fn() in a background thread while a thinking bubble spins.

    Shows a single spinner (it animates client-side) until the work finishes.
    The outcome is stored in holder as {"result": ...} or {"error": ...}.
    """
    holder.clear()

    def runner():
        try:
            holder["result"] = work_fn()
        except Exception as exc:  # surfaced to the caller via holder
            holder["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()

    chat.append(_spinner_message(caption))
    yield chat
    thread.join()
    chat.pop()


def build_generation_message(state, model):
    """Ask the model for a dataset spec, synthesize rows, and return a ChatMessage."""
    messages = [
        {"role": "system", "content": system_message},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "topic": state["topic"],
                    "complexity": state["complexity"],
                    "format": state["format"],
                    "row_count": state["rows"],
                    "minimum_tables": required_file_count(state["complexity"]),
                    "table_rule": "Return a spec only. Medium needs at least two related tables. Hard needs at least three related tables.",
                    "row_rule": "Per-table 'rows' values should sum to approximately row_count.",
                }
            ),
        },
    ]
    try:
        result = call_llm_json(messages, model)
    except Exception as exc:
        return gr.ChatMessage(
            role="assistant",
            content=f"I couldn't parse the generator response. Try again. ({exc})",
        )

    if result.get("status") != "success":
        message = (
            result.get("error", {}).get("message")
            or result.get("reply")
            or "Generation failed. Try again."
        )
        return gr.ChatMessage(role="assistant", content=message)

    data = result.get("data") or {}
    tables = data.get("tables") or []
    minimum_tables = required_file_count(state.get("complexity"))
    if len(tables) < minimum_tables:
        return gr.ChatMessage(
            role="assistant",
            content=(
                f"The generator designed {len(tables)} table(s), but {state.get('complexity')} "
                f"datasets require at least {minimum_tables} related tables. Please try again."
            ),
        )

    try:
        rendered = synthesize_tables(tables)
        files = render_files(rendered, state["format"])
    except Exception as exc:
        return gr.ChatMessage(
            role="assistant",
            content=f"I couldn't synthesize the dataset from the spec. Try again. ({exc})",
        )

    if not files:
        return gr.ChatMessage(
            role="assistant",
            content="The generator did not design any tables. Try again.",
        )

    total_rows = sum(len(table["rows"]) for table in rendered.values())
    zip_path = make_zip(files)
    reply = result.get("reply") or "Generated the dataset."
    description = data.get("description") or "Dataset files are ready."
    return gr.ChatMessage(
        role="assistant",
        content=[
            f"{reply}\n\n{description}\n\n**{total_rows:,} rows** across "
            f"{len(files)} table file(s).",
            gr.File(value=zip_path, label="Download dataset"),
        ],
    )


def stream_generation(chat, state, model):
    """Show a 'generating' spinner, then append the generated result."""
    holder = {}
    for snapshot in stream_thinking(
        chat,
        holder,
        lambda: build_generation_message(state, model),
        "Generating your dataset…",
    ):
        yield snapshot
    if "error" in holder:
        chat.append(
            gr.ChatMessage(
                role="assistant",
                content=f"I couldn't generate the dataset. Try again. ({holder['error']})",
            )
        )
    else:
        chat.append(holder["result"])
    yield chat


def clear_card_options(chat, field):
    """Remove the clickable buttons from an already-answered question card."""
    prefix = f"{field}:"
    for msg in chat:
        options = (
            msg.get("options")
            if isinstance(msg, dict)
            else getattr(msg, "options", None)
        )
        if not options:
            continue
        values = [
            (opt.get("value") if isinstance(opt, dict) else getattr(opt, "value", ""))
            or ""
            for opt in options
        ]
        if any(str(value).startswith(prefix) for value in values):
            if isinstance(msg, dict):
                msg["options"] = None
            else:
                msg.options = None


def handle_user_message(message, chat, state, model):
    chat = list(chat or [])
    state = {**{field: None for field in REQUIRED_FIELDS}, **(state or {})}
    message = (message or "").strip()
    if not message:
        yield chat, "", state
        return

    # Echo the user message and clear the textbox immediately.
    chat.append(gr.ChatMessage(role="user", content=message))
    yield chat, "", state

    # Initial loading: show a thinking spinner while the checker runs.
    checker_payload = {"current_state": state, "user_message": message}
    holder = {}
    for snapshot in stream_thinking(
        chat,
        holder,
        lambda: call_llm_json(
            [
                {"role": "system", "content": checker_message},
                {"role": "user", "content": json.dumps(checker_payload)},
            ],
            model,
        ),
        "Thinking…",
    ):
        yield snapshot, "", state
    if "error" in holder:
        chat.append(
            gr.ChatMessage(
                role="assistant",
                content=f"I couldn't understand that request. Please include a topic, complexity, format, and number of rows. ({holder['error']})",
            )
        )
        yield chat, "", state
        return
    result = holder["result"]

    topic = str(result.get("topic") or "").strip()
    if topic and topic.lower() not in {"null", "none"}:
        state["topic"] = topic
    complexity = explicit_complexity(message)
    if complexity:
        state["complexity"] = complexity
    dataset_format = explicit_format(message)
    if dataset_format:
        state["format"] = dataset_format

    rows_value = result.get("rows")
    if isinstance(rows_value, str):
        rows_value = rows_value.strip()
        rows_value = int(rows_value) if rows_value.isdigit() else None
    if isinstance(rows_value, int) and rows_value > 0:
        state["rows"] = rows_value
    rows_explicit = explicit_rows(message)
    if rows_explicit:
        state["rows"] = rows_explicit

    if not state.get("topic"):
        reply = result.get("reply") or "What dataset topic should I generate?"
        chat.append(gr.ChatMessage(role="assistant", content=reply))
        yield chat, "", state
        return

    # Ask one question at a time, step by step.
    card = next_question_card(state)
    if card:
        chat.append(card)
        yield chat, "", state
        return

    # Everything collected in one go -> generate with a live indicator.
    for snapshot in stream_generation(chat, state, model):
        yield snapshot, "", state


def capture_choice(evt: gr.SelectData):
    """Stash the clicked option's value so the chained handlers can read it.

    gr.SelectData is only delivered to the function bound directly to the
    event, not to functions chained with .then(), so we capture it here.
    """
    return str(evt.value or "")


def handle_card_choice(chat, state, model, choice):
    chat = list(chat or [])
    state = {**{field: None for field in REQUIRED_FIELDS}, **(state or {})}
    choice = str(choice or "")
    if ":" not in choice:
        yield chat, state
        return

    field, value = choice.split(":", 1)
    if field == "complexity" and value in COMPLEXITY_OPTIONS:
        state["complexity"] = value
    elif field == "format" and value in FORMAT_OPTIONS:
        state["format"] = value
    elif field == "rows" and value in ROW_OPTIONS:
        state["rows"] = int(value)
    else:
        yield chat, state
        return

    # Retire the answered card's buttons and echo the choice as a user message.
    clear_card_options(chat, field)
    chat.append(gr.ChatMessage(role="user", content=value))
    yield chat, state

    # Advance to the next missing question, if any.
    card = next_question_card(state)
    if card:
        chat.append(card)
        yield chat, state
        return

    if all(state.get(f) for f in REQUIRED_FIELDS):
        for snapshot in stream_generation(chat, state, model):
            yield snapshot, state


def lock_controls():
    """Disable input + buttons while the app is thinking or generating."""
    return tuple(gr.update(interactive=False) for _ in range(4))


def unlock_controls():
    """Re-enable input + buttons once work has finished."""
    return tuple(gr.update(interactive=True) for _ in range(4))


with gr.Blocks(title="DataAlchemy") as demo:
    gr.Markdown("# DataAlchemy")
    gr.Markdown("Generate a dataset based on the topic, complexity and format.")

    model_dropdown = gr.Dropdown(
        choices=model_names,
        value=DEFAULT_MODEL,
        label="Local model",
        info="Choose a local Ollama model to chat with.",
    )

    chatbot = gr.Chatbot(
        allow_file_downloads=True,
        height=520,
    )
    collected_state = gr.State({field: None for field in REQUIRED_FIELDS})
    pending_choice = gr.State("")

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Describe the dataset you want, e.g. retail orders, hard, CSV, 100 rows",
            show_label=False,
            scale=8,
        )
        send_button = gr.Button("Send", variant="primary", scale=1)
        clear_button = gr.Button("Clear", scale=1)

    controls = [model_dropdown, user_input, send_button, clear_button]

    send_button.click(lock_controls, outputs=controls).then(
        handle_user_message,
        inputs=[user_input, chatbot, collected_state, model_dropdown],
        outputs=[chatbot, user_input, collected_state],
    ).then(unlock_controls, outputs=controls)

    user_input.submit(lock_controls, outputs=controls).then(
        handle_user_message,
        inputs=[user_input, chatbot, collected_state, model_dropdown],
        outputs=[chatbot, user_input, collected_state],
    ).then(unlock_controls, outputs=controls)

    # Capture the SelectData first (only the direct handler receives it), then
    # lock -> handle -> unlock using the stashed choice value.
    chatbot.option_select(capture_choice, outputs=pending_choice).then(
        lock_controls, outputs=controls
    ).then(
        handle_card_choice,
        inputs=[chatbot, collected_state, model_dropdown, pending_choice],
        outputs=[chatbot, collected_state],
    ).then(
        unlock_controls, outputs=controls
    )

    clear_button.click(
        lambda: ([], "", {field: None for field in REQUIRED_FIELDS}),
        outputs=[chatbot, user_input, collected_state],
    )


demo.launch()